In [1]:
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore")


# Step 1: Load the SQuAD dataset
dataset = load_dataset("squad")

# Step 2: Extract unique contexts from the dataset
data = [item["context"] for item in dataset["train"]]
texts = list(set(data))

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


/Users/mohanreddypanga/opt/anaconda3/envs/agentic_AI/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
"""
RAG System with Binary Quantization
====================================
Binary quantization compresses each float32 vector dimension into a single bit.
- 768-dim float32 vector: 768 × 32 bits = 3072 bytes
- 768-dim binary vector:  768 × 1 bit  = 96 bytes
- That's a 32x memory reduction!

Trade-off: Slight accuracy loss, but massive speed + memory gains.
We compensate with oversampling + rescoring using original vectors.

Pipeline:
1. EmbedData       → Generate/load embeddings
2. QdrantVDB       → Store vectors WITH binary quantization
3. Retriever       → Search with quantization-aware params
4. RAGPipeline     → Full RAG: retrieve + generate answer via LLM
"""

import os
import pickle
import time
import warnings
warnings.filterwarnings("ignore")

from tqdm import tqdm
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI
from qdrant_client import QdrantClient, models

# ── Config ──
# OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY", "your-openrouter-key")
EMBED_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
BATCH_SIZE = 32
EMBED_DIM = 768
PICKLE_FILE = "embeddings_and_contexts.pkl"


# EMBED DATA

In [16]:

# ══════════════════════════════════════════════════════════════
# HELPER
# ══════════════════════════════════════════════════════════════

def batch_iterate(lst, batch_size):
    """Yield successive batches from a list."""
    for i in range(0, len(lst), batch_size):
        yield lst[i : i + batch_size]


# ══════════════════════════════════════════════════════════════
# 1. EMBED DATA
# ══════════════════════════════════════════════════════════════

class EmbedData:
    """Handles embedding model loading and batch embedding generation."""

    def __init__(self, embed_model_name=EMBED_MODEL_NAME, batch_size=BATCH_SIZE):
        self.embed_model_name = embed_model_name
        self.batch_size = batch_size
        self.embed_model = self._load_embed_model()
        self.embeddings = []
        self.contexts = []

    def _load_embed_model(self):
        print(f"Loading embedding model: {self.embed_model_name}")
        return HuggingFaceEmbedding(
            model_name=self.embed_model_name,
            trust_remote_code=True,
            cache_folder="./hf_cache",
        )

    def generate_embedding(self, context):
        return self.embed_model.get_text_embedding_batch(context)

    def embed(self, contexts):
        self.contexts = contexts
        total = (len(contexts) + self.batch_size - 1) // self.batch_size

        for batch_context in tqdm(
            batch_iterate(contexts, self.batch_size),
            total=total,
            desc="Embedding batches",
        ):
            batch_embeddings = self.generate_embedding(batch_context)
            self.embeddings.extend(batch_embeddings)

        print(f"Generated {len(self.embeddings)} embeddings")

    def save(self, filepath=PICKLE_FILE):
        with open(filepath, "wb") as f:
            pickle.dump((self.embeddings, self.contexts), f)
        print(f"Saved to {filepath}")

    def load(self, filepath=PICKLE_FILE):
        with open(filepath, "rb") as f:
            self.embeddings, self.contexts = pickle.load(f)
        print(f"Loaded {len(self.embeddings)} embeddings from {filepath}")


# Process 

```
Step 1: create_collection()
        │
        │  Creates empty collection with RULES:
        │  - vector size = 768
        │  - distance = DOT product
        │  - segments = 5
        │  - indexing = OFF
        │
        │  NO DATA YET. Just the schema/structure.
        │
        ▼
Step 2: ingest_data(embeddata)
        │
        │  Uploads your actual vectors + text:
        │
        │  Batch 1:  vectors[0:512]   + contexts[0:512]
        │  Batch 2:  vectors[512:1024] + contexts[512:1024]
        │  ...
        │  Batch 35: vectors[17920:18000] + contexts[17920:18000]
        │
        │  Then enables indexing (threshold=20000)
        │
        ▼
Step 3: search()
        │
        │  NOW you can query the collection
        │  because it has data inside.

<!-- Step 1: create_collection()
        │
        │  Creates empty collection with RULES:
        │  - vector size = 768
        │  - distance = DOT product
        │  - segments = 5
        │  - indexing = OFF
        │
        │  NO DATA YET. Just the schema/structure.
        │
        ▼
Step 2: ingest_data(embeddata)
        │
        │  Uploads your actual vectors + text:
        │
        │  Batch 1:  vectors[0:512]   + contexts[0:512]
        │  Batch 2:  vectors[512:1024] + contexts[512:1024]
        │  ...
        │  Batch 35: vectors[17920:18000] + contexts[17920:18000]
        │
        │  Then enables indexing (threshold=20000)
        │
        ▼
Step 3: search()
        │
        │  NOW you can query the collection
        │  because it has data inside. -->

# Qdrant Indexing Threshold — How It Works

## Why Two-Step Indexing?

When uploading thousands of vectors, you **don't want Qdrant rebuilding its search index after every batch**. That would be like reorganizing your entire library every time you add a single book. Instead, we:

1. **Pause indexing** during upload → fast bulk ingestion
2. **Enable indexing** after upload → build index once over all data

---

## The Two-Step Pattern in Code

```python
# STEP 1: Create collection with indexing DISABLED
self.client.create_collection(
    collection_name="my_collection",
    vectors_config=models.VectorParams(size=768, distance=models.Distance.DOT, on_disk=True),
    optimizers_config=models.OptimizersConfigDiff(
        default_segment_number=5,
        indexing_threshold=0,        # ← ZERO = never build index
    ),
)

# STEP 2: Upload all data (fast, no index rebuilds between batches)
for batch in batches:
    self.client.upload_collection(...)

# STEP 3: Enable indexing AFTER all data is uploaded
self.client.update_collection(
    collection_name="my_collection",
    optimizer_config=models.OptimizersConfigDiff(
        indexing_threshold=20000,    # ← Build index when segment has ≥ 20K vectors
    ),
)
```

---

## Visual Flow

```
indexing_threshold=0          → Pause indexing (fast uploads)
                                    │
Upload batch 1  [512 vectors] ───── │
Upload batch 2  [512 vectors] ───── │  No index rebuilds
Upload batch 3  [512 vectors] ───── │  between batches
...                                 │
Upload batch 35 [512 vectors] ───── │
                                    │
Total: ~18,000 vectors uploaded     │
                                    │
indexing_threshold=20000      → Build index ONCE (efficient)
                                    │
                                    ▼
                    Qdrant checks each segment:
                                    │
              ┌─────────────────────┼─────────────────────┐
              ▼                     ▼                     ▼
     Segment 1: 3,600       Segment 2: 3,600      Segment 5: 3,600
     3,600 < 20,000         3,600 < 20,000        3,600 < 20,000
     → Brute force ✓        → Brute force ✓       → Brute force ✓

     (No HNSW index built — segments too small, brute force is fast enough)
```

---

## What Happens with a Larger Dataset?

```
     IF dataset had 500,000 vectors instead:

              ┌─────────────────────┼─────────────────────┐
              ▼                     ▼                     ▼
     Segment 1: 100,000     Segment 2: 100,000    Segment 5: 100,000
     100,000 > 20,000       100,000 > 20,000      100,000 > 20,000
     → Build HNSW index ✓   → Build HNSW index ✓  → Build HNSW index ✓

     (HNSW index built — brute force too slow for 100K per segment)
```

---

## What is `indexing_threshold`?

It's the **minimum number of vectors per segment** before Qdrant builds an HNSW search index on that segment.

| Value | Meaning |
|-------|---------|
| `0` | Never build index (disable indexing completely) |
| `5000` | Build HNSW when segment has ≥ 5,000 vectors |
| `20000` | Build HNSW when segment has ≥ 20,000 vectors |

**Important:** The threshold is checked **per segment**, not total. With `default_segment_number=5` and 18,000 total vectors:

```
18,000 vectors ÷ 5 segments = ~3,600 vectors per segment
3,600 < 20,000 → No HNSW index → Brute force search
```

---

## What is `default_segment_number=5`?

Qdrant splits your collection into multiple **segments** (like partitions). Each segment is an independent searchable unit.

```
┌────────────────── Collection ──────────────────┐
│                                                 │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐       │
│  │Segment 1 │ │Segment 2 │ │Segment 3 │       │
│  │ 3,600    │ │ 3,600    │ │ 3,600    │       │
│  │ vectors  │ │ vectors  │ │ vectors  │ ...   │
│  └──────────┘ └──────────┘ └──────────┘       │
│                                                 │
│  ┌──────────┐ ┌──────────┐                     │
│  │Segment 4 │ │Segment 5 │                     │
│  │ 3,600    │ │ 3,600    │                     │
│  │ vectors  │ │ vectors  │                     │
│  └──────────┘ └──────────┘                     │
│                                                 │
│  Total: ~18,000 vectors across 5 segments       │
└─────────────────────────────────────────────────┘
```

**Why multiple segments?**
- Parallel search across segments → faster
- Independent indexing per segment
- Better memory management

---

## Brute Force vs HNSW Index

| Method | How It Works | Speed | When Used |
|--------|-------------|-------|-----------|
| **Brute force** | Compare query against EVERY vector | Slow for large data, fine for small | Segment < threshold |
| **HNSW index** | Graph-based approximate search, only visits a subset of vectors | Fast even for millions | Segment ≥ threshold |

```
Brute Force (3,600 vectors):
Query → Compare with ALL 3,600 → Return top 5
         ~3,600 comparisons (fast enough)

HNSW Index (100,000 vectors):
Query → Navigate graph → Visit ~200 nodes → Return top 5
         ~200 comparisons instead of 100,000 (massive speedup)
```

---

## Choosing the Right Threshold

| Dataset Size | Recommended Threshold | Reason |
|-------------|----------------------|--------|
| < 10K vectors | `20000` (or any high number) | Brute force is fast enough |
| 10K - 100K | `5000` - `10000` | HNSW helps but segments aren't huge |
| 100K - 1M | `5000` | Need HNSW for reasonable speed |
| 1M+ | `1000` - `5000` | Definitely need HNSW everywhere |

**Rule of thumb:** If brute force search takes > 100ms, lower the threshold to trigger HNSW indexing.

---

## Without vs With the Two-Step Pattern

```
❌ WITHOUT (indexing_threshold = default during upload):

Upload batch 1  → Qdrant rebuilds index  (slow)
Upload batch 2  → Qdrant rebuilds index  (slow)
Upload batch 3  → Qdrant rebuilds index  (slow)
...
Upload batch 35 → Qdrant rebuilds index  (slow)

Total: 35 index rebuilds = VERY SLOW ingestion


✅ WITH (indexing_threshold = 0 during upload):

Upload batch 1  → No indexing  (fast)
Upload batch 2  → No indexing  (fast)
Upload batch 3  → No indexing  (fast)
...
Upload batch 35 → No indexing  (fast)
Enable indexing → Build ONCE   (efficient)

Total: 0 rebuilds during upload + 1 final build = FAST ingestion
```

---

## Summary

1. **Set `indexing_threshold=0`** before uploading → disables indexing for fast bulk ingestion
2. **Upload all vectors** in batches without any index overhead
3. **Set `indexing_threshold=20000`** after uploading → Qdrant builds HNSW index once
4. The threshold is **per segment** — only segments exceeding the threshold get an HNSW index
5. Small segments use **brute force** (fine for < 20K vectors per segment)
6. Large segments use **HNSW** (necessary for 100K+ vectors per segment)

# QDRNT

In [31]:

# ══════════════════════════════════════════════════════════════
# 2. QDRANT VECTOR DB — STANDARD (No Quantization)
# ══════════════════════════════════════════════════════════════

class QdrantVDB:
    """Standard Qdrant vector database — full precision vectors."""

    def __init__(self, collection_name, vector_dim=EMBED_DIM, batch_size=512):
        self.collection_name = collection_name
        self.batch_size = batch_size
        self.vector_dim = vector_dim

    def define_client(self):
        self.client = QdrantClient(url="http://localhost:6333", prefer_grpc=True)
        print("Connected to Qdrant (standard mode)")

    def create_collection(self):
        if not self.client.collection_exists(self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=models.VectorParams(
                    size=self.vector_dim,
                    distance=models.Distance.DOT,
                    on_disk=True,
                ),
                optimizers_config=models.OptimizersConfigDiff(
                    default_segment_number=5,
                    indexing_threshold=0,
                ),
            )
            print(f"Created collection: {self.collection_name}")
        else:
            print(f"Collection {self.collection_name} already exists")

    def ingest_data(self, embeddata):
        total = len(embeddata.contexts) // self.batch_size

        for batch_context, batch_embeddings in tqdm(
            zip(
                batch_iterate(embeddata.contexts, self.batch_size),
                batch_iterate(embeddata.embeddings, self.batch_size),
            ),
            total=total,
            desc="Ingesting (standard)",
        ):
            self.client.upload_collection(
                collection_name=self.collection_name,
                vectors=batch_embeddings,
                payload=[{"context": ctx} for ctx in batch_context],
            )

        self.client.update_collection(
            collection_name=self.collection_name,
            optimizer_config=models.OptimizersConfigDiff(indexing_threshold=20000),
        )

        count = self.client.get_collection(self.collection_name).points_count
        print(f"Ingested {count} vectors (standard)")


In [32]:

# ══════════════════════════════════════════════════════════════
# 3. QDRANT VECTOR DB — BINARY QUANTIZATION
# ══════════════════════════════════════════════════════════════

class QdrantVDB_BQ:
    """
    Qdrant with Binary Quantization.
    
    How it works:
    ┌─────────────────────────────────────────────────────────┐
    │  Original Vector (float32)                              │
    │  [0.12, -0.45, 0.78, -0.33, 0.56, ...]                │
    │  Each dimension = 32 bits → 768 × 32 = 24,576 bits     │
    │                                                         │
    │              Binary Quantization                        │
    │              ─────────────────────                      │
    │  Rule: positive → 1, negative → 0                      │
    │                                                         │
    │  Binary Vector                                          │
    │  [1, 0, 1, 0, 1, ...]                                  │
    │  Each dimension = 1 bit → 768 × 1 = 768 bits           │
    │                                                         │
    │  Memory: 24,576 bits → 768 bits = 32x reduction!       │
    │  Search: Hamming distance (XOR + popcount) = ultra fast │
    └─────────────────────────────────────────────────────────┘
    
    Accuracy recovery via oversampling + rescoring:
    1. Search binary vectors (fast) → get 2x-4x candidates
    2. Rescore candidates using original float32 vectors (accurate)
    3. Return top-k from rescored results
    """

    def __init__(self, collection_name, vector_dim=EMBED_DIM, batch_size=512):
        self.collection_name = collection_name
        self.batch_size = batch_size
        self.vector_dim = vector_dim

    def define_client(self):
        self.client = QdrantClient(url="http://localhost:6333", prefer_grpc=True)
        print("Connected to Qdrant (binary quantization mode)")

    def create_collection(self):
        if not self.client.collection_exists(self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=models.VectorParams(
                    size=self.vector_dim,
                    distance=models.Distance.DOT,
                    on_disk=True,
                ),
                optimizers_config=models.OptimizersConfigDiff(
                    default_segment_number=5,
                    indexing_threshold=0,
                ),
                # ┌──────────────────────────────────────────┐
                # │  THIS IS THE KEY DIFFERENCE              │
                # │  Enable binary quantization              │
                # │                                          │
                # │  always_ram=True keeps binary vectors     │
                # │  in RAM for fast search, while original   │
                # │  float32 vectors stay on disk             │
                # └──────────────────────────────────────────┘
                quantization_config=models.BinaryQuantization(
                    binary=models.BinaryQuantizationConfig(
                        always_ram=True,
                    ),
                ),
            )
            print(f"Created collection with BINARY QUANTIZATION: {self.collection_name}")
        else:
            print(f"Collection {self.collection_name} already exists")

    def ingest_data(self, embeddata):
        total = len(embeddata.contexts) // self.batch_size

        for batch_context, batch_embeddings in tqdm(
            zip(
                batch_iterate(embeddata.contexts, self.batch_size),
                batch_iterate(embeddata.embeddings, self.batch_size),
            ),
            total=total,
            desc="Ingesting (binary quantized)",
        ):
            self.client.upload_collection(
                collection_name=self.collection_name,
                vectors=batch_embeddings,
                payload=[{"context": ctx} for ctx in batch_context],
            )

        self.client.update_collection(
            collection_name=self.collection_name,
            optimizer_config=models.OptimizersConfigDiff(indexing_threshold=20000),
        )

        count = self.client.get_collection(self.collection_name).points_count
        print(f"Ingested {count} vectors (binary quantized)")


# Retriever

In [33]:


# ══════════════════════════════════════════════════════════════
# 4. RETRIEVER — STANDARD
# ══════════════════════════════════════════════════════════════

class Retriever:
    """Standard retriever — searches full precision vectors."""

    def __init__(self, vector_db, embeddata):
        self.vector_db = vector_db
        self.embeddata = embeddata

    def search(self, query, top_k=5):
        query_embedding = self.embeddata.embed_model.get_query_embedding(query)

        start_time = time.time()

        result = self.vector_db.client.query_points(
            collection_name=self.vector_db.collection_name,
            query=query_embedding,
            limit=top_k,
            timeout=1000,
        )

        elapsed = time.time() - start_time
        print(f"Standard search: {elapsed:.4f}s")

        # Make sure we return the points list, not a tuple
        if hasattr(result, 'points'):
            return result.points
        else:
            return result

        return result


# ══════════════════════════════════════════════════════════════
# 5. RETRIEVER — BINARY QUANTIZATION
# ══════════════════════════════════════════════════════════════

class RetrieverBQ:
    """
    Binary quantization retriever with oversampling + rescoring.
    
    Search flow:
    ┌─────────────────────────────────────────────────────────┐
    │  Step 1: Fast binary search                             │
    │  ─────────────────────────                              │
    │  Query vector → binarize → Hamming distance search      │
    │  Retrieve top_k × oversampling candidates (e.g. 5×2=10)│
    │                                                         │
    │  Step 2: Rescore with original vectors                  │
    │  ────────────────────────────────────                   │
    │  Load float32 vectors for 10 candidates                 │
    │  Compute exact DOT product similarity                   │
    │  Return top 5 from rescored results                     │
    │                                                         │
    │  Result: Nearly same accuracy as full search,           │
    │          but much faster initial retrieval               │
    └─────────────────────────────────────────────────────────┘
    """

    def __init__(self, vector_db, embeddata):
        self.vector_db = vector_db
        self.embeddata = embeddata

    def search(self, query, top_k=5):
        query_embedding = self.embeddata.embed_model.get_query_embedding(query)

        start_time = time.time()

        result = self.vector_db.client.query_points(
            collection_name=self.vector_db.collection_name,
            query=query_embedding,
            limit=top_k,
            search_params=models.SearchParams(
                quantization=models.QuantizationSearchParams(
                    ignore=False,   # USE quantization (don't ignore it)
                    rescore=True,   # Rescore with original vectors for accuracy
                    oversampling=2.0,  # Fetch 2x candidates, then rescore
                )
            ),
            timeout=1000,
        )

        elapsed = time.time() - start_time
        print(f"Binary quantized search: {elapsed:.4f}s")
        # Make sure we return the points list, not a tuple
        if hasattr(result, 'points'):
            return result.points
        else:
            return result

        return result

In [34]:

# ══════════════════════════════════════════════════════════════
# 6. RAG PIPELINE
# ══════════════════════════════════════════════════════════════
OPENROUTER_KEY = os.environ["OPENROUTER_API_KEY"]

class RAGPipeline:
    """Full RAG: retrieve contexts + generate answer via LLM."""

    def __init__(self, retriever):
        self.retriever = retriever
        self.llm = OpenAI(
            model="gpt-4o-mini",
            api_base="https://openrouter.ai/api/v1",
            api_key=OPENROUTER_KEY,
            temperature=0,
        )

    def query(self, question, top_k=5):
        # Retrieve
        results = self.retriever.search(question, top_k=top_k)

        # Extract contexts
        contexts = [r.payload["context"] for r in results]
        context_str = "\n\n---\n\n".join(contexts)

        # Generate
        prompt = (
            "You are a helpful assistant. Answer based ONLY on the context below. "
            "If the answer isn't in the context, say 'I don't know.'\n\n"
            f"Context:\n{context_str}\n\n"
            f"Question: {question}\n"
            "Answer: "
        )

        response = self.llm.complete(prompt)
        return {
            "question": question,
            "answer": str(response),
            "contexts": contexts,
            "num_results": len(results),
        }


# ══════════════════════════════════════════════════════════════
# 7. BENCHMARK — COMPARE STANDARD vs BINARY QUANTIZATION
# ══════════════════════════════════════════════════════════════

def benchmark(retriever_standard, retriever_bq, queries, top_k=5):
    """Compare search speed between standard and binary quantized retrieval."""

    print("\n" + "=" * 60)
    print("BENCHMARK: Standard vs Binary Quantization")
    print("=" * 60)

    standard_times = []
    bq_times = []

    for query in queries:
        print(f"\nQuery: {query}")

        # Standard search
        t1 = time.time()
        r1 = retriever_standard.search(query, top_k)
        t_std = time.time() - t1
        standard_times.append(t_std)

        # Binary quantized search
        t2 = time.time()
        r2 = retriever_bq.search(query, top_k)
        t_bq = time.time() - t2
        bq_times.append(t_bq)

        print(f"  Standard: {t_std:.4f}s | BQ: {t_bq:.4f}s | Speedup: {t_std/t_bq:.2f}x")

    avg_std = sum(standard_times) / len(standard_times)
    avg_bq = sum(bq_times) / len(bq_times)

    print(f"\n{'='*60}")
    print(f"AVERAGE — Standard: {avg_std:.4f}s | BQ: {avg_bq:.4f}s | Speedup: {avg_std/avg_bq:.2f}x")
    print(f"{'='*60}")



# RUN PIPLINES

In [35]:


# ── Step 1: Load embeddings ──
embeddata = EmbedData(batch_size=BATCH_SIZE)

if os.path.exists(PICKLE_FILE):
    embeddata.load(PICKLE_FILE)
else:
    from datasets import load_dataset
    dataset = load_dataset("rajpurkar/squad", split="train")
    contexts = list(set(dataset["context"]))
    embeddata.embed(contexts)
    embeddata.save(PICKLE_FILE)

print(f"Total vectors: {len(embeddata.embeddings)}")
print(f"Vector dimension: {len(embeddata.embeddings[0])}")

# ── Step 2: Setup STANDARD database ──
print("\n--- Setting up STANDARD collection ---")
db_standard = QdrantVDB("squad_standard")
db_standard.define_client()
db_standard.create_collection()
db_standard.ingest_data(embeddata)

# ── Step 3: Setup BINARY QUANTIZED database ──
print("\n--- Setting up BINARY QUANTIZED collection ---")
db_bq = QdrantVDB_BQ("squad_binary_quantized")
db_bq.define_client()
db_bq.create_collection()
db_bq.ingest_data(embeddata)

# ── Step 4: Create retrievers ──
retriever_standard = Retriever(db_standard, embeddata)
retriever_bq = RetrieverBQ(db_bq, embeddata)

# ── Step 5: Benchmark ──
test_queries = [
    "What is the capital of France?",
    "Who developed the theory of relativity?",
    "What causes earthquakes?",
    "When was the United Nations founded?",
    "What is photosynthesis?",
]

benchmark(retriever_standard, retriever_bq, test_queries)

# ── Step 6: Full RAG query ──
print("\n\n--- Full RAG Pipeline (Binary Quantized) ---")
rag = RAGPipeline(retriever_bq)

for q in test_queries[:2]:
    result = rag.query(q)
    print(f"\nQ: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Contexts retrieved: {result['num_results']}")

Loading embedding model: nomic-ai/nomic-embed-text-v1.5


<All keys matched successfully>


Loaded 18891 embeddings from embeddings_and_contexts.pkl
Total vectors: 18891
Vector dimension: 768

--- Setting up STANDARD collection ---
Connected to Qdrant (standard mode)
Collection squad_standard already exists


Ingesting (standard): 37it [00:07,  4.80it/s]                        


Ingested 94455 vectors (standard)

--- Setting up BINARY QUANTIZED collection ---
Connected to Qdrant (binary quantization mode)
Collection squad_binary_quantized already exists


Ingesting (binary quantized): 37it [00:04,  8.00it/s]                        


Ingested 94455 vectors (binary quantized)

BENCHMARK: Standard vs Binary Quantization

Query: What is the capital of France?
Standard search: 0.1547s
Binary quantized search: 0.0247s
  Standard: 1.8311s | BQ: 0.1038s | Speedup: 17.65x

Query: Who developed the theory of relativity?
Standard search: 0.0452s
Binary quantized search: 0.0134s
  Standard: 0.1109s | BQ: 0.1042s | Speedup: 1.06x

Query: What causes earthquakes?
Standard search: 0.0506s
Binary quantized search: 0.0084s
  Standard: 0.1435s | BQ: 0.0676s | Speedup: 2.12x

Query: When was the United Nations founded?
Standard search: 0.0372s
Binary quantized search: 0.0169s
  Standard: 0.0981s | BQ: 0.1972s | Speedup: 0.50x

Query: What is photosynthesis?
Standard search: 0.0336s
Binary quantized search: 0.0097s
  Standard: 0.1298s | BQ: 0.0800s | Speedup: 1.62x

AVERAGE — Standard: 0.4627s | BQ: 0.1105s | Speedup: 4.19x


--- Full RAG Pipeline (Binary Quantized) ---
Binary quantized search: 0.0077s

Q: What is the capital of Fran

In [38]:
# ── Step 6: Full RAG query ──
print("\n\n--- Full RAG Pipeline (Binary VS Regular Quantized) ---")
rag_BN = RAGPipeline(retriever_bq)
rag_regular = RAGPipeline(retriever_standard)
for q in test_queries:
    result_BN = rag_BN.query(q)
    result_regular = rag_regular.query(q)
    print(f"\nQ: {result_BN['question']}")
    print(f"A BN: {result_BN['answer']}")
    print(f"A REGULAR: {result_regular['answer']}")
    print(f"Contexts retrieved from BN: {result_BN['num_results']}")
    print(f"Contexts retrieved from regular: {result_BN['num_results']}")



--- Full RAG Pipeline (Binary VS Regular Quantized) ---
Binary quantized search: 0.0393s
Standard search: 0.0372s

Q: What is the capital of France?
A BN: I don't know.
A REGULAR: I don't know.
Contexts retrieved from BN: 5
Contexts retrieved from regular: 5
Binary quantized search: 0.0127s
Standard search: 0.0321s

Q: Who developed the theory of relativity?
A BN: Albert Einstein developed the theory of relativity.
A REGULAR: Albert Einstein developed the theory of relativity.
Contexts retrieved from BN: 5
Contexts retrieved from regular: 5
Binary quantized search: 0.0331s
Standard search: 0.1094s

Q: What causes earthquakes?
A BN: I don't know.
A REGULAR: I don't know.
Contexts retrieved from BN: 5
Contexts retrieved from regular: 5
Binary quantized search: 0.0296s
Standard search: 0.0987s

Q: When was the United Nations founded?
A BN: I don't know.
A REGULAR: I don't know.
Contexts retrieved from BN: 5
Contexts retrieved from regular: 5
Binary quantized search: 0.0237s
Standard sea